In [ ]:
# @title **Luau 1.5B Unsloth Distillation & GGUF Export**

# @markdown ---
# @markdown ### **GitHub Repository (Source Code):**
GITHUB_REPO_URL = "https://github.com/bananamort/luau-qwen2.5-coder-1.5b-distillation.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

# @markdown ### **Hugging Face Hub & IO:**
HF_TOKEN = "" # @param {type:"string"}
DATASET_REPO_ID = "bananamort/the-luau-stack-fim-tokenized" # @param {type:"string"}
DATASET_FILENAME = "fim_train.parquet" # @param {type:"string"}
UPLOAD_MODEL_REPO_ID = "bananamort/Luau-Qwen2.5-1.5B-FIM" # @param {type:"string"}

# @markdown ### **Student Model (Hui et al. 2024, arXiv:2409.12186):**
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct" # @param {type:"string"}
MAX_SEQ_LENGTH = 2048 # @param {type:"integer"}
LORA_R = 64 # @param {type:"integer"}
LORA_ALPHA = 64 # @param {type:"integer"}
USE_RSLORA = True # @param {type:"boolean"} # rsLoRA scaling: alpha / sqrt(r) (Kalajdzievski 2023, arXiv:2312.03732)

# @markdown ### **Teacher Model (Williams 2025, TorpedoSoftware/Luau-Qwen3-4B-FIM-v0.1):**
TEACHER_MODEL_NAME = "TorpedoSoftware/Luau-Qwen3-4B-FIM-v0.1" # @param {type:"string"}
DISTILL_TEMPERATURE = 2.0 # @param {type:"number"} # Softmax temperature scaling (Hinton et al. 2015, arXiv:1503.02531)
DISTILL_ALPHA = 0.5 # @param {type:"number"} # Dual-objective loss balance (Sanh et al. 2019, arXiv:1910.01108)

# @markdown ### **Training Parameters:**
BATCH_SIZE = 2 # @param {type:"integer"}
GRAD_ACCUM = 8 # @param {type:"integer"}
LEARNING_RATE = 2e-4 # @param {type:"number"}
NUM_TRAIN_EPOCHS = 1 # @param {type:"integer"}
WARMUP_RATIO = 0.03 # @param {type:"number"}
OPTIMIZER = "paged_adamw_8bit" # @param ["paged_adamw_8bit", "adamw_8bit", "adamw_torch"]
LOG_STEPS = 25 # @param {type:"integer"}
SAVE_STEPS = 1000 # @param {type:"integer"}
PUSH_TO_HUB = True # @param {type:"boolean"}
RESUME_FROM_CHECKPOINT = False # @param {type:"boolean"}

# @markdown ### **Export Options:**
SAVE_16BIT_MERGED = True # @param {type:"boolean"}
EXPORT_GGUF = True # @param {type:"boolean"}
QUANT_METHOD = "q4_0" # @param ["q4_0", "q4_k_m", "q8_0", "f16"]
QAT_SCHEME = "int4" # @param ["int4", ""]

# @markdown ### **Weights & Biases (WandB):**
USE_WANDB = True # @param {type:"boolean"}
WANDB_TOKEN = "" # @param {type:"string"}
WANDB_PROJECT = "luau-1.5b-distill" # @param {type:"string"}

# @markdown ### **Runtime Management:**
AUTO_DISCONNECT_VM = True # @param {type:"boolean"}

import os
import subprocess

try:
    # 1. Git Setup
    if os.path.exists("repo"):
        %cd repo
        !git pull origin {BRANCH}
    elif os.path.exists(".git"):
        !git pull origin {BRANCH}
    elif GITHUB_REPO_URL.strip():
        print(f"Cloning repository: {GITHUB_REPO_URL} (branch: {BRANCH})...")
        !git clone --depth 1 -b {BRANCH} {GITHUB_REPO_URL.strip()} repo
        %cd repo

    # 2. Install Dependencies
    print("Installing Unsloth and training dependencies...")
    !pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/flash_attn-2.7.3+cu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
    !pip install -q --no-deps trl peft accelerate bitsandbytes datasets huggingface_hub pyarrow wandb "torchao>=0.16.0"

    # 3. Build CLI Command
    cmd = [
        "python", "-u", "src/train.py",
        "--dataset_repo_id", DATASET_REPO_ID.strip(),
        "--dataset_filename", DATASET_FILENAME.strip(),
        "--upload_model_repo_id", UPLOAD_MODEL_REPO_ID.strip(),
        "--model_name", MODEL_NAME.strip(),
        "--teacher_model", TEACHER_MODEL_NAME.strip(),
        "--max_seq_length", str(MAX_SEQ_LENGTH),
        "--lora_r", str(LORA_R),
        "--lora_alpha", str(LORA_ALPHA),
        "--temperature", str(DISTILL_TEMPERATURE),
        "--alpha", str(DISTILL_ALPHA),
        "--chunk_size", "2048",
        "--batch_size", str(BATCH_SIZE),
        "--grad_accum", str(GRAD_ACCUM),
        "--learning_rate", str(LEARNING_RATE),
        "--epochs", str(NUM_TRAIN_EPOCHS),
        "--warmup_ratio", str(WARMUP_RATIO),
        "--optimizer", OPTIMIZER.strip(),
        "--log_steps", str(LOG_STEPS),
        "--save_steps", str(SAVE_STEPS),
        "--quant_method", QUANT_METHOD.strip(),
    ]

    if HF_TOKEN.strip():
        cmd.extend(["--token", HF_TOKEN.strip()])

    if not USE_RSLORA:
        cmd.append("--no-use_rslora")

    if not SAVE_16BIT_MERGED:
        cmd.append("--no-save_16bit_merged")

    if not EXPORT_GGUF:
        cmd.append("--no-export_gguf")

    if PUSH_TO_HUB:
        cmd.append("--push_to_hub")

    if RESUME_FROM_CHECKPOINT:
        cmd.extend(["--resume_from_checkpoint", "True"])

    if QAT_SCHEME.strip():
        cmd.extend(["--qat_scheme", QAT_SCHEME.strip()])

    if USE_WANDB and WANDB_TOKEN.strip():
        cmd.extend([
            "--use_wandb",
            "--wandb_token", WANDB_TOKEN.strip(),
            "--wandb_project", WANDB_PROJECT.strip(),
        ])

    # 4. Execute with Real-Time Streaming
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.stdout.close()
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

except Exception as e:
    print(f"\nERROR: {e}")
    raise
finally:
    if AUTO_DISCONNECT_VM:
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception:
            pass
